# 06. Linear Regression (Ordinary Least Squares) Model
**Crop Yield Prediction and Forecasting Research Project**  
*Author: Aryan Puranik*

This notebook implements the first predictive machine learning model in our sequence: **Ordinary Least Squares (OLS) Linear Regression** for next-year crop yield forecasting ($kg/ha$).

### Research Objective & Hypotheses
Our objective is to forecast $\text{Yield}_{r,c,t+1}$ at forecast origin year $t$ using the frozen historical feature set (**Spec A: Historical-Only**, 31 features).

- **Hypothesis 1 ($H_1$ - Multi-variable Value-Add):** Does an unregularized multi-variable linear model combining crop-region dummy fixed effects, exact calendar lags, rolling averages, and causal climate anomalies outperform our best persistence heuristic baseline (EWMA Recency-Weighted MAE = 305.71 kg/ha)?
- **Hypothesis 2 ($H_2$ - Multicollinearity & Variance Inflation):** Because historical lags ($t-1, t-2, t-3$) and rolling summaries are mutually correlated ($r > 0.85$), will unregularized OLS assign unstable, counter-intuitive oscillating coefficients (+/- signs) and inflate parameter variance across 300+ geographic dummy variables?
- **Hypothesis 3 ($H_3$ - Heteroscedasticity):** Are prediction residuals normally distributed and homoscedastic across the entire range of crop yield scales, or does error magnitude expand with higher-yielding crops (necessitating tree-based or regularized models)?

### Research Protocol
1. **Leak-Free Preprocessing:** Fit `ColumnTransformer` (median imputation + `StandardScaler` for numericals; most-frequent imputation + `OneHotEncoder(handle_unknown='ignore')` for categoricals) strictly on training data ($t+1 \le 2011$).
2. **Inner Chronological CV:** Evaluate out-of-fold stability using 3 expanding-window folds before touching validation data.
3. **Dual-Cohort Validation:** Evaluate on the untouched 2012–2014 validation partition across both the **Full Validation Cohort** ($N=10,610$) and the **Complete History Cohort** ($N=9,966$).
4. **Comprehensive Evaluation:** MAE, RMSE, $R^2$, MAP, WAPE, 5-tier quintile confusion matrix, residual diagnostics, crop-wise breakdown, and leaderboard update.

## 1. Setup, Environment, and Reproducibility

We import standard libraries, configure headless plotting, set random seeds, and load the frozen modeling splits exported from Phase 04.

In [1]:
from pathlib import Path
import json
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from sklearn.linear_model import LinearRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    confusion_matrix, classification_report, accuracy_score
)

sns.set_theme(style="whitegrid")
RANDOM_SEED = 42

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports" / "model_evaluation"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Load frozen splits from Phase 04
train_df = pd.read_csv(PROCESSED_DIR / "train_final.csv")
val_df = pd.read_csv(PROCESSED_DIR / "val_final.csv")

# Load frozen feature specification (Spec A: Historical-Only Yield Forecasting)
with open(PROCESSED_DIR / "spec_A_historical_only.json", "r") as f:
    spec = json.load(f)

numerical_features = spec["numerical_features"]
categorical_features = spec["categorical_features"]
target_col = spec["target"]
target_year_col = spec["target_year"]

print(f"Loaded Training Data:   {len(train_df):,} rows | Target_Years: {train_df[target_year_col].min()} - {train_df[target_year_col].max()}")
print(f"Loaded Validation Data: {len(val_df):,} rows | Target_Years: {val_df[target_year_col].min()} - {val_df[target_year_col].max()}")
print(f"Complete History Cohort in Validation: {(val_df['History_3yr_Complete'] == 1).sum():,} rows")
print(f"Numerical Features ({len(numerical_features)}): {numerical_features}")
print(f"Categorical Features ({len(categorical_features)}): {categorical_features}")
print(f"Target Column: '{target_col}'")

Loaded Training Data:   164,915 rows | Target_Years: 1967 - 2011
Loaded Validation Data: 10,610 rows | Target_Years: 2012 - 2014
Complete History Cohort in Validation: 9,966 rows
Numerical Features (28): ['Year', 'Is_Post_1990', 'Temperature', 'Rainfall', 'Humidity', 'Temperature_Anomaly', 'Rainfall_Anomaly', 'Humidity_Anomaly', 'pH', 'Nitrogen', 'Phosphorus', 'Potassium', 'Pesticide', 'NPK_Total', 'N_P_Ratio', 'Rain_Temp_Ratio', 'Yield_Lag1', 'Yield_Lag2', 'Yield_Lag3', 'Yield_Lag5', 'Yield_Rolling3', 'Yield_Rolling3_Std', 'Yield_Change_5Y', 'Yield_Trend_Velocity', 'Hist_Region_Crop_Median', 'Yield_Relative_Index', 'History_Lag1_Available', 'History_3yr_Complete']
Categorical Features (3): ['Region', 'Crop', 'Soil Type']
Target Column: 'Target_Yield'


## 2. Leak-Free Preprocessing & Pipeline Construction

We construct an encapsulated `Pipeline`:
- **Numerical Features:** Imputed with median and standardized using `StandardScaler`.
- **Categorical Features:** Imputed with most frequent value and one-hot encoded using `OneHotEncoder(handle_unknown='ignore', sparse_output=True)`.
- **Estimator:** Ordinary Least Squares `LinearRegression()` solving the normal equation $\hat{\beta} = (X^T X)^{-1} X^T y$ via sparse SVD/QR decomposition.

In [2]:
preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numerical_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=True))
    ]), categorical_features)
])

linear_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

feature_cols = categorical_features + numerical_features
print(f"Constructed leak-free Pipeline: {len(feature_cols)} input features -> ColumnTransformer -> LinearRegression()")

Constructed leak-free Pipeline: 31 input features -> ColumnTransformer -> LinearRegression()


## 3. Inner Chronological Cross-Validation (3 Folds)

Before evaluating on the holdout validation period (2012–2014), we test model stability on training-era data using 3 expanding-window chronological splits:
- **Fold 1:** Train $\le 2005 \implies$ Validate 2006–2007
- **Fold 2:** Train $\le 2007 \implies$ Validate 2008–2009
- **Fold 3:** Train $\le 2009 \implies$ Validate 2010–2011

In [3]:
cv_splits = [
    ("Fold 1 (Train <= 2005 | Val 2006-2007)", train_df["Target_Year"] <= 2005, (train_df["Target_Year"] >= 2006) & (train_df["Target_Year"] <= 2007)),
    ("Fold 2 (Train <= 2007 | Val 2008-2009)", train_df["Target_Year"] <= 2007, (train_df["Target_Year"] >= 2008) & (train_df["Target_Year"] <= 2009)),
    ("Fold 3 (Train <= 2009 | Val 2010-2011)", train_df["Target_Year"] <= 2009, (train_df["Target_Year"] >= 2010) & (train_df["Target_Year"] <= 2011))
]

cv_records = []
for fold_name, tr_mask, va_mask in cv_splits:
    tr_sub = train_df[tr_mask]
    va_sub = train_df[va_mask]
    
    linear_pipeline.fit(tr_sub[feature_cols], tr_sub[target_col])
    preds = linear_pipeline.predict(va_sub[feature_cols])
    
    mae = mean_absolute_error(va_sub[target_col], preds)
    rmse = np.sqrt(mean_squared_error(va_sub[target_col], preds))
    r2 = r2_score(va_sub[target_col], preds)
    wape = 100.0 * np.sum(np.abs(va_sub[target_col] - preds)) / np.sum(va_sub[target_col])
    
    cv_records.append({
        "Fold": fold_name,
        "Train_Rows": len(tr_sub),
        "Val_Rows": len(va_sub),
        "MAE (kg/ha)": round(mae, 2),
        "RMSE (kg/ha)": round(rmse, 2),
        "R2": round(r2, 4),
        "WAPE (%)": round(wape, 2)
    })

cv_df = pd.DataFrame(cv_records)
print("=== INNER CHRONOLOGICAL CROSS-VALIDATION RESULTS ===")
display(cv_df)
print(f"Mean Inner CV MAE: {cv_df['MAE (kg/ha)'].mean():.2f} kg/ha | Mean R2: {cv_df['R2'].mean():.4f}")

=== INNER CHRONOLOGICAL CROSS-VALIDATION RESULTS ===
                                     Fold  Train_Rows  Val_Rows  MAE (kg/ha)  RMSE (kg/ha)      R2  WAPE (%)
0  Fold 1 (Train <= 2005 | Val 2006-2007)      141669      8045       266.96        799.59  0.7687     19.08
1  Fold 2 (Train <= 2007 | Val 2008-2009)      149714      7657       255.93        537.17  0.8714     17.97
2  Fold 3 (Train <= 2009 | Val 2010-2011)      157371      7544       284.53        531.84  0.8959     18.02
Mean Inner CV MAE: 269.14 kg/ha | Mean R2: 0.8453


## 4. Full Model Fitting & In-Sample Training Fit

We fit the pipeline on all training records ($N=164,915$, Target_Year $\le 2011$) and compute in-sample goodness-of-fit metrics to establish baseline bias vs. variance.

In [4]:
t0 = time.time()
linear_pipeline.fit(train_df[feature_cols], train_df[target_col])
train_time = time.time() - t0

train_preds = linear_pipeline.predict(train_df[feature_cols])
train_mae = mean_absolute_error(train_df[target_col], train_preds)
train_rmse = np.sqrt(mean_squared_error(train_df[target_col], train_preds))
train_r2 = r2_score(train_df[target_col], train_preds)
valid_mask_tr = train_df[target_col] >= 10.0
train_map = 100.0 * np.mean(np.abs((train_df[target_col].values[valid_mask_tr] - train_preds[valid_mask_tr]) / train_df[target_col].values[valid_mask_tr]))
train_wape = 100.0 * np.sum(np.abs(train_df[target_col] - train_preds)) / np.sum(train_df[target_col])

print(f"Full Linear Regression Pipeline fitted in {train_time:.2f} seconds.")
print(f"In-Sample Training Fit (N = {len(train_df):,}):")
print(f"  MAE:  {train_mae:.2f} kg/ha")
print(f"  RMSE: {train_rmse:.2f} kg/ha")
print(f"  R2:   {train_r2:.4f}")
print(f"  MAP:  {train_map:.2f}%")
print(f"  WAPE: {train_wape:.2f}%")

Full Linear Regression Pipeline fitted in 4.86 seconds.
In-Sample Training Fit (N = 164,915):
  MAE:  224.89 kg/ha
  RMSE: 446.81 kg/ha
  R2:   0.9104
  MAP:  32.39%
  WAPE: 18.58%


## 5. Dual-Cohort Validation Inference (Unseen 2012–2014) & Benchmark Comparison

We generate next-year yield forecasts on the 2012–2014 validation set and evaluate both cohorts:
1. **Full Validation Cohort ($N=10,610$):** All district-crop pairs present in 2012–2014, including rows with imputed historical lags.
2. **Complete History Cohort ($N=9,966$):** The rigorous benchmark cohort where 3 consecutive past historical years are observed.

We directly compare Linear Regression against the persistence baselines established in Notebook 05.

In [5]:
val_preds = linear_pipeline.predict(val_df[feature_cols])

def evaluate_cohort(y_true, y_pred, cohort_name):
    err = y_true - y_pred
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    valid_mask = y_true >= 10.0
    map_err = 100.0 * np.mean(np.abs(err[valid_mask] / y_true[valid_mask]))
    wape = 100.0 * np.sum(np.abs(err)) / np.sum(y_true)
    return {
        "Cohort": cohort_name,
        "Rows": len(y_true),
        "MAE (kg/ha)": round(mae, 2),
        "RMSE (kg/ha)": round(rmse, 2),
        "R2": round(r2, 4),
        "MAP (%)": round(map_err, 2),
        "WAPE (%)": round(wape, 2)
    }

comp_mask = (val_df["History_3yr_Complete"] == 1).values
full_metrics = evaluate_cohort(val_df[target_col].values, val_preds, "Full Validation (2012-2014)")
comp_metrics = evaluate_cohort(val_df[target_col].values[comp_mask], val_preds[comp_mask], "Complete History Cohort (N=9,966)")

val_perf_df = pd.DataFrame([full_metrics, comp_metrics])
print("=== LINEAR REGRESSION VALIDATION PERFORMANCE ===")
display(val_perf_df)

# Direct comparison against baseline leaderboard
leaderboard_path = REPORTS_DIR / "model_comparison_leaderboard.csv"
if leaderboard_path.exists():
    lb_existing = pd.read_csv(leaderboard_path)
    print("\n=== BENCHMARK COMPARISON (COMPLETE HISTORY COHORT) ===")
    comp_summary = pd.DataFrame({
        "Model": lb_existing["Model"].tolist() + ["Linear Regression (OLS)"],
        "MAE (kg/ha)": lb_existing["Complete_Hist_MAE"].tolist() + [comp_metrics["MAE (kg/ha)"]],
        "RMSE (kg/ha)": lb_existing["Complete_Hist_RMSE"].tolist() + [comp_metrics["RMSE (kg/ha)"]],
        "R2": lb_existing["Complete_Hist_R2"].tolist() + [comp_metrics["R2"]],
        "MAP (%)": lb_existing["Complete_Hist_MAP"].tolist() + [comp_metrics["MAP (%)"]]
    }).sort_values("MAE (kg/ha)")
    display(comp_summary)

=== LINEAR REGRESSION VALIDATION PERFORMANCE ===
                              Cohort   Rows  MAE (kg/ha)  RMSE (kg/ha)      R2  MAP (%)  WAPE (%)
0        Full Validation (2012-2014)  10610       325.94        563.49  0.8932    33.30     19.95
1  Complete History Cohort (N=9,966)   9966       323.86        556.75  0.8965    32.78     19.57

=== BENCHMARK COMPARISON (COMPLETE HISTORY COHORT) ===
                                             Model  MAE (kg/ha)  RMSE (kg/ha)      R2  MAP (%)
3  Baseline: EWMA Recency-Weighted (0.5, 0.3, 0.2)       305.71        546.29  0.9004    28.63
2                    Baseline: Rolling 3-Year Mean       314.66        562.34  0.8944    29.39
4                          Linear Regression (OLS)       323.86        556.75  0.8965    32.78
1              Baseline: Naive Persistence (Lag 1)       328.35        593.71  0.8823    29.90
0          Baseline: Region-Crop Historical Median       588.30        962.24  0.6909    40.05


## 6. Model Coefficients & Multicollinearity Analysis

We inspect the learned OLS weights to test **Hypothesis 2 ($H_2$)**.
When predictors are mutually correlated (such as `Yield_Lag1`, `Yield_Lag2`, and `Yield_Rolling3`), unregularized OLS assigns large opposing positive and negative weights to fit sample noise, leading to coefficient inflation.

In [6]:
cat_encoder = linear_pipeline.named_steps["preprocessor"].named_transformers_["cat"].named_steps["encoder"]
encoded_cat_names = list(cat_encoder.get_feature_names_out(categorical_features))
all_features = numerical_features + encoded_cat_names
coefficients = linear_pipeline.named_steps["regressor"].coef_
intercept = linear_pipeline.named_steps["regressor"].intercept_

coef_df = pd.DataFrame({
    "Feature": all_features,
    "Coefficient": coefficients,
    "Abs_Weight": np.abs(coefficients)
}).sort_values("Abs_Weight", ascending=False).reset_index(drop=True)

print(f"Model Intercept: {intercept:.2f} kg/ha")
print(f"Total Feature Parameters: {len(coef_df)} (Numerical: {len(numerical_features)}, Dummy Fixed Effects: {len(encoded_cat_names)})")

print("\nTop 10 Positive Coefficients:")
display(coef_df.sort_values("Coefficient", ascending=False).head(10))

print("\nTop 10 Negative Coefficients:")
display(coef_df.sort_values("Coefficient", ascending=True).head(10))

print("\nNumerical Feature Coefficients (Standardized Inputs):")
display(coef_df[coef_df["Feature"].isin(numerical_features)].sort_values("Abs_Weight", ascending=False))

Model Intercept: 1210.93 kg/ha
Total Feature Parameters: 364 (Numerical: 28, Dummy Fixed Effects: 336)

Top 10 Positive Coefficients:
                             Feature  Coefficient   Abs_Weight
0        Region_Chamoli, Uttarakhand  2074.877941  2074.877941
1    Region_Uttar Kashi, Uttarakhand  1909.078425  1909.078425
2     Region_Pithorgarh, Uttarakhand  1646.133220  1646.133220
3   Region_Chamba, Himachal  Pradesh  1132.489930  1132.489930
4   Region_Shimla, Himachal  Pradesh   821.082412   821.082412
5                         Yield_Lag1   812.886963   812.886963
6  Region_Tehri Garhwal, Uttarakhand   803.835374   803.835374
7     Region_Darjeeling, West Bengal   668.948607   668.948607
8        Region_Almorah, Uttarakhand   644.913668   644.913668
9       Region_Nainital, Uttarakhand   609.957394   609.957394

Top 10 Negative Coefficients:
                              Feature  Coefficient  Abs_Weight
10         Region_Mayurbhanja, Odisha  -543.829368  543.829368
11              

## 7. Binned Quintile Confusion Matrix & Categorical Accuracy

To evaluate how well the model predicts operational yield tiers, we discretize continuous actual and predicted yields into 5 ordered quintiles using training-derived thresholds (`Very Low`, `Low`, `Medium`, `High`, `Very High`).
We measure exact Tier Accuracy and Adjacent Tier Accuracy (predictions within $\pm 1$ tier).

In [7]:
quantiles = [0.2, 0.4, 0.6, 0.8]
q_cuts = list(np.quantile(train_df[target_col].values, quantiles))
tier_labels = ["Very Low", "Low", "Medium", "High", "Very High"]
bins = [-np.inf] + q_cuts + [np.inf]

y_true_tiers = pd.cut(val_df[target_col], bins=bins, labels=tier_labels)
y_pred_tiers = pd.cut(val_preds, bins=bins, labels=tier_labels)

tier_acc = accuracy_score(y_true_tiers, y_pred_tiers)
cm = confusion_matrix(y_true_tiers, y_pred_tiers, labels=tier_labels)
cm_df = pd.DataFrame(cm, index=[f"Actual {t}" for t in tier_labels], columns=[f"Pred {t}" for t in tier_labels])

adj_correct = sum(cm[i, j] for i in range(5) for j in range(5) if abs(i - j) <= 1)
adj_acc = adj_correct / len(val_df)

print(f"Yield Quintile Classification Accuracy: {tier_acc * 100:.2f}%")
print(f"Adjacent Tier Accuracy (+/- 1 Tier):     {adj_acc * 100:.2f}%")

plt.figure(figsize=(8, 6))
sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.title(f"Linear Regression Yield Tier Confusion Matrix (Accuracy: {tier_acc*100:.2f}%)", fontsize=12)
plt.xlabel("Predicted Yield Tier")
plt.ylabel("Actual Yield Tier")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "linear_regression_confusion_matrix.png", dpi=300)
plt.close()

print("\nClassification Report across Tiers:")
print(classification_report(y_true_tiers, y_pred_tiers))

Yield Quintile Classification Accuracy: 66.01%
Adjacent Tier Accuracy (+/- 1 Tier):     95.30%

Classification Report across Tiers:
              precision    recall  f1-score   support

        High       0.56      0.66      0.61      2420
         Low       0.55      0.53      0.54      1591
      Medium       0.48      0.57      0.52      1884
   Very High       0.89      0.84      0.86      3596
    Very Low       0.77      0.43      0.55      1119

    accuracy                           0.66     10610
   macro avg       0.65      0.60      0.62     10610
weighted avg       0.68      0.66      0.66     10610



## 8. Residual Diagnostics & Heteroscedasticity Analysis

To test **Hypothesis 3 ($H_3$)**, we inspect residual distributions, heteroscedasticity across predicted yield ranges, and temporal error stability across validation years (2012, 2013, 2014).

In [8]:
residuals = val_df[target_col].values - val_preds
val_df_diag = val_df.copy()
val_df_diag["Residual"] = residuals
val_df_diag["Predicted"] = val_preds

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Actual vs. Predicted Scatter
axes[0, 0].scatter(val_preds, val_df[target_col], alpha=0.15, s=8, color="#1f77b4")
max_v = max(val_df[target_col].max(), val_preds.max())
axes[0, 0].plot([0, max_v], [0, max_v], "r--", linewidth=1.5, label="Identity (Perfect Fit)")
axes[0, 0].set_title(f"Actual vs. Predicted Yield (R2 = {full_metrics['R2']:.4f})")
axes[0, 0].set_xlabel("Predicted Next-Year Yield (kg/ha)")
axes[0, 0].set_ylabel("Actual Next-Year Yield (kg/ha)")
axes[0, 0].set_xlim(0, 12000)
axes[0, 0].set_ylim(0, 12000)
axes[0, 0].legend()

# 2. Residual Distribution
sns.histplot(residuals, bins=60, kde=True, ax=axes[0, 1], color="#2ca02c")
axes[0, 1].axvline(0, color="red", linestyle="--", label="Zero Residual")
axes[0, 1].axvline(np.mean(residuals), color="black", linestyle=":", label=f"Mean Residual ({np.mean(residuals):.1f})")
axes[0, 1].set_title("Residual Distribution (Actual - Predicted)")
axes[0, 1].set_xlabel("Residual (kg/ha)")
axes[0, 1].set_xlim(-2500, 2500)
axes[0, 1].legend()

# 3. Residuals vs. Predicted (Heteroscedasticity check)
axes[1, 0].scatter(val_preds, residuals, alpha=0.15, s=8, color="#ff7f0e")
axes[1, 0].axhline(0, color="red", linestyle="--")
axes[1, 0].set_title("Residuals vs. Fitted Values (Variance Check)")
axes[1, 0].set_xlabel("Predicted Next-Year Yield (kg/ha)")
axes[1, 0].set_ylabel("Residual (kg/ha)")
axes[1, 0].set_xlim(0, 12000)
axes[1, 0].set_ylim(-4000, 4000)

# 4. Residuals by Target Year
sns.boxplot(data=val_df_diag, x="Target_Year", y="Residual", hue="Target_Year", legend=False, ax=axes[1, 1], palette="Set2")
axes[1, 1].axhline(0, color="red", linestyle="--")
axes[1, 1].set_title("Residual Stability Across Validation Years (2012-2014)")
axes[1, 1].set_xlabel("Target Year")
axes[1, 1].set_ylabel("Residual (kg/ha)")
axes[1, 1].set_ylim(-2500, 2500)

plt.tight_layout()
plt.savefig(REPORTS_DIR / "linear_regression_residual_diagnostics.png", dpi=300)
plt.close()

## 9. Crop-Wise Performance Breakdown

We inspect crop-specific errors to evaluate localized model behavior across varied agricultural scales (e.g., Sugarcane vs. Pulses).

In [9]:
val_df_diag["Abs_Error"] = np.abs(residuals)

crop_summary = val_df_diag.groupby("Crop").apply(lambda g: pd.Series({
    "Samples": len(g),
    "Mean Actual (kg/ha)": g[target_col].mean(),
    "MAE (kg/ha)": mean_absolute_error(g[target_col], g["Predicted"]),
    "RMSE (kg/ha)": np.sqrt(mean_squared_error(g[target_col], g["Predicted"])),
    "R2": r2_score(g[target_col], g["Predicted"]),
    "WAPE (%)": 100.0 * np.sum(np.abs(g[target_col] - g["Predicted"])) / np.sum(g[target_col])
})).reset_index().sort_values("Samples", ascending=False)

print("=== CROP-WISE VALIDATION PERFORMANCE ===")
display(crop_summary.round(2))

top_crops = crop_summary.sort_values("R2", ascending=False).head(3)["Crop"].tolist()
worst_crops = crop_summary.sort_values("R2", ascending=True).head(3)["Crop"].tolist()
print(f"\nTop 3 Crops by R2 Fit: {top_crops}")
print(f"Lowest 3 Crops by R2 Fit: {worst_crops}")

=== CROP-WISE VALIDATION PERFORMANCE ===
                  Crop  Samples  Mean Actual (kg/ha)  MAE (kg/ha)  RMSE (kg/ha)    R2  WAPE (%)
12                Rice    806.0              2342.70       329.01        463.49  0.76     14.04
7                Maize    779.0              2590.92       617.61        937.26  0.63     23.84
20               Wheat    720.0              2406.30       397.84        553.71  0.73     16.53
2             Chickpea    711.0               941.16       240.62        324.12  0.30     25.57
10           Pigeonpea    702.0               857.89       218.07        306.07  0.50     25.42
14             Sesamum    696.0               411.29       116.58        150.67  0.63     28.35
18           Sugarcane    675.0              6288.99       974.52       1420.88  0.74     15.50
5            Groundnut    636.0              1441.73       251.25        363.67  0.69     17.43
11  Rapeseed & Mustard    634.0               896.90       173.46        243.22  0.68     19.34

## 10. Model Artifacts Export & Leaderboard Update

We persist all model artifacts and record the official performance of Linear Regression on `reports/model_evaluation/model_comparison_leaderboard.csv`.

In [10]:
# 1. Save fitted pipeline
pipeline_path = MODELS_DIR / "linear_regression_pipeline.joblib"
joblib.dump(linear_pipeline, pipeline_path)
print(f"Saved fitted pipeline to: {pipeline_path}")

# 2. Save evaluation metrics JSON
metrics_payload = {
    "model_name": "Linear Regression (OLS)",
    "specification": spec["specification_name"],
    "features_total": len(feature_cols),
    "training_time_seconds": round(train_time, 2),
    "training_fit": {
        "rows": len(train_df),
        "mae": round(train_mae, 2),
        "rmse": round(train_rmse, 2),
        "r2": round(train_r2, 4),
        "map": round(train_map, 2),
        "wape": round(train_wape, 2)
    },
    "inner_cv_mean": {
        "mae": round(cv_df["MAE (kg/ha)"].mean(), 2),
        "r2": round(cv_df["R2"].mean(), 4)
    },
    "full_validation": full_metrics,
    "complete_history_validation": comp_metrics,
    "tier_classification": {
        "accuracy": round(tier_acc * 100, 2),
        "adjacent_accuracy": round(adj_acc * 100, 2)
    }
}
metrics_path = REPORTS_DIR / "linear_regression_metrics.json"
with open(metrics_path, "w") as f:
    json.dump(metrics_payload, f, indent=2)
print(f"Saved metrics to: {metrics_path}")

# 3. Save validation predictions
val_preds_export = val_df[["Region", "Crop", "Year", "Target_Year", target_col]].copy()
val_preds_export["Predicted_Yield"] = val_preds
val_preds_export["Residual"] = residuals
val_preds_export["Actual_Tier"] = np.asarray(y_true_tiers)
val_preds_export["Predicted_Tier"] = np.asarray(y_pred_tiers)
preds_path = REPORTS_DIR / "linear_regression_val_predictions.csv"
val_preds_export.to_csv(preds_path, index=False)
print(f"Saved validation predictions to: {preds_path}")

# 4. Save confusion matrix CSV
cm_path = REPORTS_DIR / "linear_regression_confusion_matrix.csv"
cm_df.to_csv(cm_path)
print(f"Saved confusion matrix to: {cm_path}")

# 5. Update Leaderboard
leaderboard_row = {
    "Model": "Linear Regression (OLS)",
    "Full_Val_MAE": full_metrics["MAE (kg/ha)"],
    "Full_Val_RMSE": full_metrics["RMSE (kg/ha)"],
    "Full_Val_R2": full_metrics["R2"],
    "Full_Val_MAP": full_metrics["MAP (%)"],
    "Complete_Hist_MAE": comp_metrics["MAE (kg/ha)"],
    "Complete_Hist_RMSE": comp_metrics["RMSE (kg/ha)"],
    "Complete_Hist_R2": comp_metrics["R2"],
    "Complete_Hist_MAP": comp_metrics["MAP (%)"],
    "Tier_Accuracy": round(tier_acc * 100, 2)
}

if leaderboard_path.exists():
    lb_df = pd.read_csv(leaderboard_path)
    lb_df = lb_df[lb_df["Model"] != "Linear Regression (OLS)"]
    lb_df = pd.concat([lb_df, pd.DataFrame([leaderboard_row])], ignore_index=True)
else:
    lb_df = pd.DataFrame([leaderboard_row])

lb_df.to_csv(leaderboard_path, index=False)
print(f"\nUpdated Leaderboard ({leaderboard_path}):")
display(lb_df)

Saved fitted pipeline to: D:\crop_yield_prediction\models\linear_regression_pipeline.joblib
Saved metrics to: D:\crop_yield_prediction\reports\model_evaluation\linear_regression_metrics.json
Saved validation predictions to: D:\crop_yield_prediction\reports\model_evaluation\linear_regression_val_predictions.csv
Saved confusion matrix to: D:\crop_yield_prediction\reports\model_evaluation\linear_regression_confusion_matrix.csv

Updated Leaderboard (D:\crop_yield_prediction\reports\model_evaluation\model_comparison_leaderboard.csv):
                                             Model  Full_Val_MAE  Full_Val_RMSE  Full_Val_R2  Full_Val_MAP  Complete_Hist_MAE  Complete_Hist_RMSE  Complete_Hist_R2  Complete_Hist_MAP  Tier_Accuracy
0          Baseline: Region-Crop Historical Median        586.81         971.59       0.6824         40.18             588.30              962.24            0.6909              40.05          55.42
1              Baseline: Naive Persistence (Lag 1)           NaN     

## 11. Scientific Summary & Research Conclusions

### Research Findings
1. **Rejection of Hypothesis 1 ($H_1$):** On the Complete History Cohort ($N=9,966$), unregularized OLS achieved an MAE of **323.86 kg/ha** ($R^2 = 0.8965$). While this improves upon naive 1-year persistence (328.35 kg/ha), it **fails to outperform** the simple EWMA recency-weighted baseline (**305.71 kg/ha**, $R^2 = 0.9004$) and the 3-year rolling mean (**314.66 kg/ha**).
2. **Confirmation of Hypothesis 2 ($H_2$ - Multicollinearity & Variance Inflation):** Coefficient inspection reveals severe parameter instability. `Yield_Lag1` (+812.87), `Yield_Lag2` (+293.99), and `Yield_Lag3` (+217.17) carry high positive weights, while `Yield_Rolling3` is assigned a large negative weight (-275.15) to cancel collinearity. Simultaneously, 309 regional dummy fixed effects take on extreme weights (up to $\pm 2,075$ kg/ha), demonstrating unpenalized variance.
3. **Confirmation of Hypothesis 3 ($H_3$ - Heteroscedasticity):** Residual diagnostics show variance expanding significantly for high-yield predictions (>4,000 kg/ha). Furthermore, tier classification reveals 66.01% exact quintile accuracy, but 95.30% adjacent tier accuracy.

### Direct Scientific Next Step: Ridge Regression ($L_2$ Regularization)
The empirical evidence demonstrates that unregularized OLS suffers from variance inflation due to collinear predictors and over-parameterized geographic fixed effects. In Notebook 07 (`07_ridge_regression.ipynb`), we test whether adding an $L_2$ penalty (Tikhonov regularization) stabilizes coefficient weights, shrinks noisy dummy effects toward zero, and improves out-of-sample forecasting accuracy.